In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.dim_employees AS
SELECT 
  employee_key,
  trim(employee_full_name) AS employee_full_name,
  
  -- Rozbicie imienia i nazwiska (pomocne do sortowania i szukania w Power BI)
  element_at(split(trim(employee_full_name), ' '), 1) AS first_name,
  element_at(split(trim(employee_full_name), ' '), -1) AS last_name,
  
  upper(trim(work_center)) AS work_center,
  date_of_employment,
  date_of_leaving,
  
  -- Flaga aktywności pracownika na dzień dzisiejszy
  CASE 
    WHEN date_of_employment IS NOT NULL 
     AND (date_of_leaving IS NULL OR date_of_leaving >= CURRENT_DATE()) 
    THEN TRUE 
    ELSE FALSE 
  END AS is_active,
  
  -- Wyliczony staż pracy w miesiącach (dla pracujących do dziś, dla byłych do date_of_leaving)
  timestampdiff(
    MONTH, 
    date_of_employment, 
    COALESCE(date_of_leaving, CURRENT_DATE())
  ) AS tenure_months,
  
  _silver_ingested_at,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM data_warehouse_factory.silver.silver_employees;